In [ ]:
## includes some code from cellPLM paper

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath("../../models/cellPLM"))

import warnings
warnings.filterwarnings("ignore")

import hdf5plugin
import numpy as np
import anndata as ad
from scipy.sparse import csr_matrix
from CellPLM.utils import set_seed
from CellPLM.pipeline.cell_embedding import CellEmbeddingPipeline
import scanpy as sc
import matplotlib.pyplot as plt
import rapids_singlecell as rsc  # For faster evaluation, we recommend the installation of rapids_singlecell.

In [4]:
PRETRAIN_VERSION = '20231027_85M'
DEVICE = 'cuda:0'
set_seed(42)

2026-04-26 20:38:55 | [INFO] Setting global random seed to 42


In [3]:
data = ad.read_h5ad('../../data/cellPLM/data/gse155468.h5ad')
data.obs_names_make_unique()

In [5]:
pipeline = CellEmbeddingPipeline(pretrain_prefix=PRETRAIN_VERSION, # Specify the pretrain checkpoint to load
                                 pretrain_directory='../../data/cellPLM/ckpt')
pipeline.model

OmicsFormer(
  (embedder): OmicsEmbeddingLayer(
    (act): ReLU()
    (norm0): GroupNorm(4, 1024, eps=1e-05, affine=True)
    (dropout): Dropout(p=0.2, inplace=False)
    (extra_linear): Sequential(
      (0): Linear(in_features=1024, out_features=1024, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.2, inplace=False)
      (3): GroupNorm(4, 1024, eps=1e-05, affine=True)
    )
    (pe_enc): Sinusoidal2dPE(
      (pe_enc): Embedding(10000, 1024)
    )
    (feat_enc): OmicsEmbedder()
  )
  (mask_model): MaskBuilder()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x FlowformerLayer(
        (self_attn): Flow_Attention(
          (query_projection): Linear(in_features=1024, out_features=1024, bias=True)
          (key_projection): Linear(in_features=1024, out_features=1024, bias=True)
          (value_projection): Linear(in_features=1024, out_features=1024, bias=True)
          (out_projection): Linear(in_features=1024, out_features=1024, bias=True)
         

In [6]:
embedding = pipeline.predict(data, device=DEVICE) # Specify a gpu or cpu for model inference

2026-04-26 20:40:27 | [INFO] querying 1-1000 ...


Automatically converting gene symbols to ensembl ids...


2026-04-26 20:40:28 | [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-04-26 20:40:29 | [INFO] querying 1001-2000 ...
2026-04-26 20:40:30 | [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-04-26 20:40:31 | [INFO] querying 2001-3000 ...
2026-04-26 20:40:32 | [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-04-26 20:40:33 | [INFO] querying 3001-4000 ...
2026-04-26 20:40:34 | [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-04-26 20:40:35 | [INFO] querying 4001-5000 ...
2026-04-26 20:40:36 | [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-04-26 20:40:37 | [INFO] querying 5001-6000 ...
2026-04-26 20:40:38 | [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-04-26 20:40:39 | [INFO] querying 6001-7000 ...
2026-04-26 20:40:40 | [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-04-26 20:40

After filtering, 9933 genes remain.


In [8]:
embedding.shape

torch.Size([48082, 512])

In [28]:
def summarize(a):
    import scipy.sparse as sp
    print(f"shape: {a.shape} (cells × genes)")
    print(f"X: {type(a.X).__name__}, dtype={a.X.dtype}, "
          f"density={(a.X.nnz/(a.n_obs*a.n_vars)):.4f}" if sp.issparse(a.X) else "")
    s = a.X[:min(2000, a.n_obs)]
    s = s.toarray() if sp.issparse(s) else s
    print(f"X range: [{s.min():.2f}, {s.max():.2f}], mean={s.mean():.3f}")
    print(f"obs cols: {a.obs.columns.tolist()}")
    print(f"var cols: {a.var.columns.tolist()}")
    print(f"var_names sample: {a.var_names[:5].tolist()}")
    print(f"obsm: {list(a.obsm.keys())}")
    print(f"layers: {list(a.layers.keys())}")
    print(f"uns: {list(a.uns.keys())}")

In [29]:
summarize(data)

shape: (48082, 12382) (cells × genes)
X: csr_matrix, dtype=int64, density=0.1332
X range: [0.00, 2714.00], mean=0.647
obs cols: ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'stim', 'integrated_snn_res.0.6', 'seurat_clusters', 'celltype', 'celltype2']
var cols: []
var_names sample: ['AL627309.1', 'AL669831.5', 'LINC00115', 'FAM41C', 'NOC2L']
obsm: []
layers: []
uns: []


In [31]:
# write embedding to h5ad
embedding_adata = ad.AnnData(X=embedding.cpu().numpy())
embedding_adata.obs_names = data.obs_names
embedding_adata.var_names = [f"dim_{i}" for i in range(embedding.shape[1])]
embedding_adata.write_h5ad('gse155468_embedding.h5ad')